## 0) Configuration (edit these paths first)

# UFM-Transformer Kaggle Orchestration Notebook

This notebook orchestrates the project execution based on `instructions_integration.txt`.

✅ **Step 2 is skipped** (dataset download), because the dataset is assumed to already exist in the Kaggle environment.


In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil
import os
import itertools

# =========================
# USER CONFIGURATION
# =========================
# Path where this repository is available in Kaggle (Working or Input).
# Example: Path('/kaggle/working/Ngbayafou_Deep Learning Biométrie Multimodale')
PROJECT_ROOT = Path('/kaggle/input/datasets/lynettecheripha/ufm-transformer-seperate')

# ---------------------------------------------------------------------------
# Dataset configuration
# ---------------------------------------------------------------------------

# Combined dataset: face + fingerprint share the same subject structure
DATASET_PATH = Path('/kaggle/input/datasets/lynettecheripha/ufm-transformer-seperate/data')

# Outputs in Kaggle working directory
OUTPUT_DIR = Path('/kaggle/working/output')
RESULTS_DIR = Path('/kaggle/working/results')

# Training/eval parameters
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4
IMAGE_SIZE = 224
EMBED_DIM = 512
NUM_WORKERS = 4
DEVICE = 'auto'  # auto, cuda, cpu

# Resume configuration for training step
# Set RESUME_FROM manually to a checkpoint path, or keep None and AUTO_RESUME=True.
RESUME_FROM = Path('/kaggle/working/output/best_model_phase2.pt')
AUTO_RESUME = False

# -------------------------------------------------------------
# Timeout: gracefully stop training before Kaggle's 30h limit.
# The training process will receive a SIGTERM, finish the
# current epoch, save a checkpoint, and exit cleanly.
# Set to None to disable (run until natural completion).
TIMEOUT_SECONDS = 29 * 3600  # 29 hours = 104400 seconds

SRC_DIR = Path('/kaggle/input/datasets/lynettecheripha/ufm-code-ddp-3')
LATEX_DIR = Path('/kaggle/working/latex')
TEMPLATE_LATEX_DIR = PROJECT_ROOT / 'latex'

def run(cmd, cwd=None, check=True):
    print('\n' + '='*100)
    print('RUN:', ' '.join(str(x) for x in cmd))
    print('CWD:', cwd if cwd else os.getcwd())
    print('='*100)
    result = subprocess.run(cmd, cwd=cwd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with return code {result.returncode}: {cmd}')
    return result.returncode

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC_DIR:', SRC_DIR)
print('DATASET_PATH:', DATASET_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('RESULTS_DIR:', RESULTS_DIR)
print('TEMPLATE_LATEX_DIR:', TEMPLATE_LATEX_DIR)

# Copy LaTeX template directory to working directory so it is writable
source_latex = TEMPLATE_LATEX_DIR
if source_latex.exists():
    print(f'Copying latex directory from {source_latex} to {LATEX_DIR}...')
    if LATEX_DIR.exists():
        shutil.rmtree(LATEX_DIR)
    shutil.copytree(source_latex, LATEX_DIR)
    print('Successfully copied latex directory.')
else:
    print(f'Source latex directory {source_latex} not found.')

In [ ]:
# Safety switch: set to True only when you really want to clear OUTPUT_DIR.
# CLEAR_OUTPUT_DIR = False

# if CLEAR_OUTPUT_DIR:
#     if OUTPUT_DIR.exists():
#         print('Clearing OUTPUT_DIR:', OUTPUT_DIR)
#         for item in OUTPUT_DIR.iterdir():
#             if item.is_dir():
#                 shutil.rmtree(item)
#             else:
#                 item.unlink()
#     OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#     print('OUTPUT_DIR cleared.')
# else:
#     print('CLEAR_OUTPUT_DIR is False; keeping existing OUTPUT_DIR contents.')
#     if OUTPUT_DIR.exists():
#         existing = sorted(OUTPUT_DIR.iterdir())
#         print(f'Existing items: {len(existing)}')
#         for item in existing[:20]:
#             print(' -', item)
#     else:
#         print('OUTPUT_DIR does not exist yet:', OUTPUT_DIR)


In [ ]:
# Move/Copy pre-trained checkpoints from input to working directory
import shutil
from pathlib import Path

source_dir = Path('/kaggle/input/datasets/lynettecheripha/ufm-outputs')
target_dir = Path('/kaggle/working/output')

if source_dir.exists():
    print(f'Copying checkpoints from {source_dir} to {target_dir}...')
    target_dir.mkdir(parents=True, exist_ok=True)
    copied_count = 0
    for item in source_dir.glob('**/*'):
        if item.is_file():
            rel_path = item.relative_to(source_dir)
            dest_file = target_dir / rel_path
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, dest_file)
            copied_count += 1
            print(f'  Copied: {rel_path}')
    print(f'Successfully copied {copied_count} files.')
else:
    print(f'Source checkpoints directory {source_dir} not found. Skipping.')


## 1) Analyze and verify project structure

In [ ]:
required_paths = [
    PROJECT_ROOT,
    SRC_DIR,
    SRC_DIR / 'main.py',
    SRC_DIR / 'requirements.txt',
    LATEX_DIR,
    TEMPLATE_LATEX_DIR,
]

for p in required_paths:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')

print('\nTop-level content:')
for p in sorted(PROJECT_ROOT.iterdir()):
    print(' -', p.name)

print('\nSource files in src/:')
for p in sorted(SRC_DIR.glob('*.py')):
    print(' -', p.name)


## 2) Environment setup (Step 1 from the guide)

In [ ]:
# Install project dependencies (Kaggle-safe)
# We intentionally avoid upgrading pip/system packages because Kaggle
# preinstalls RAPIDS components that may report resolver conflicts.
rc = run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--upgrade-strategy', 'only-if-needed',
    '-r', str(SRC_DIR / 'requirements.txt')
], check=False)
if rc != 0:
    raise RuntimeError('Dependency installation failed. See logs above.')

print('Note: pip dependency-conflict warnings with RAPIDS can be ignored unless you use cudf/cuml/dask-cuda in this notebook.')

# Quick torch/device check
run([
    sys.executable, '-c',
    'import os, torch; '
    'print(torch.__version__); '
    'print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES")); '
    'print("CUDA available:", torch.cuda.is_available()); '
    'print("GPU count:", torch.cuda.device_count()); '
    '[print(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())]'
])

## 3) Dataset check (Step 2 skipped by request)

We only verify that the dataset path exists and contains files.

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset path not found: {DATASET_PATH}')

print('Dataset path exists:', DATASET_PATH)
sample_files = list(itertools.islice(DATASET_PATH.rglob('*'), 30))
print(f'Showing up to 30 entries ({len(sample_files)} found):')
for p in sample_files:
    print(' -', p)

## 4) Run training (Step 3)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

resume_from = RESUME_FROM
if resume_from is None and AUTO_RESUME:
    import re

    def checkpoint_rank(path):
        match = re.fullmatch(r'checkpoint_phase([12])_epoch(\d+)\.pt', path.name)
        if match:
            phase = int(match.group(1))
            epoch = int(match.group(2))
            if epoch <= 0:
                return -1, -1, -1
            return phase, epoch, 2
        match = re.fullmatch(r'best_model_phase([12])\.pt', path.name)
        if match:
            return int(match.group(1)), 0, 1
        return -1, -1, -1

    candidates = [
        p for p in OUTPUT_DIR.glob('*.pt')
        if checkpoint_rank(p) != (-1, -1, -1)
    ]
    resume_from = max(candidates, key=checkpoint_rank) if candidates else None

train_cmd = [
    sys.executable, 'main.py',
    '--mode', 'train',
    '--dataset_path', str(DATASET_PATH),
    '--output_dir', str(OUTPUT_DIR),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--lr', str(LR),
    '--weight_decay', str(WEIGHT_DECAY),
    '--image_size', str(IMAGE_SIZE),
    '--embed_dim', str(EMBED_DIM),
    '--num_workers', str(NUM_WORKERS),
    '--device', DEVICE,
]

if TIMEOUT_SECONDS is not None:
    train_cmd += ['--timeout', str(TIMEOUT_SECONDS)]

if resume_from is not None:
    resume_from = Path(resume_from)
    if resume_from.exists():
        print('Resuming training from:', resume_from)
        train_cmd += ['--resume', str(resume_from)]
    else:
        raise FileNotFoundError(f'RESUME_FROM checkpoint not found: {resume_from}')
else:
    print('No checkpoint selected; starting training from scratch.')

import time
import subprocess as _sp

if TIMEOUT_SECONDS is not None:
    print(f'Timeout monitoring: {TIMEOUT_SECONDS}s (≈{TIMEOUT_SECONDS/3600:.1f}h)')
    GRACE_PERIOD = 300  # 5min for the subprocess to save & exit
    start_time = time.time()

    proc = _sp.Popen(train_cmd, cwd=str(SRC_DIR), text=True)

    try:
        while proc.poll() is None:
            elapsed = time.time() - start_time
            if elapsed >= TIMEOUT_SECONDS:
                print(f'\n*** TIMEOUT ({TIMEOUT_SECONDS}s) — sending SIGTERM ***')
                proc.terminate()
                try:
                    proc.wait(timeout=GRACE_PERIOD)
                except _sp.TimeoutExpired:
                    print('*** Grace period expired — sending SIGKILL ***')
                    proc.kill()
                    proc.wait()
                break
            time.sleep(10)
    except KeyboardInterrupt:
        print('\n*** Notebook interrupted — forwarding SIGTERM ***')
        proc.terminate()
        try:
            proc.wait(timeout=GRACE_PERIOD)
        except _sp.TimeoutExpired:
            proc.kill()
            proc.wait()
        raise

    rc = proc.returncode
    if rc != 0:
        print(f'Training process exited with code {rc}')
    else:
        print('Training process exited successfully.')
else:
    rc = run(train_cmd, cwd=str(SRC_DIR), check=False)
    if rc != 0:
        print(f'Training process exited with code {rc} (continuing with saved artifacts).')


## 5) Run evaluation (Step 4)

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Resolve best checkpoint: prefer phase2, fall back to phase1
_ckpt_candidates = [
    OUTPUT_DIR / 'best_model_phase2.pt',
    OUTPUT_DIR / 'best_model_phase1.pt',
    OUTPUT_DIR / 'shutdown_checkpoint_phase2.pt',
    OUTPUT_DIR / 'shutdown_checkpoint_phase1.pt',
]
best_checkpoint = next((p for p in _ckpt_candidates if p.exists()), None)
if best_checkpoint is None:
    # last resort: any .pt in OUTPUT_DIR
    _pts = sorted(OUTPUT_DIR.glob('*.pt'))
    best_checkpoint = _pts[-1] if _pts else None
if best_checkpoint is None:
    raise FileNotFoundError(f'No checkpoint found in {OUTPUT_DIR}. Run training first.')
print(f'Using checkpoint: {best_checkpoint}')

# Use evaluate.py directly — it produces figures/ and tables/ with PDFs + .tex
eval_cmd = [
    sys.executable, 'evaluate.py',
    '--model_path', str(best_checkpoint),
    '--dataset_path', str(DATASET_PATH),
    '--output_dir', str(RESULTS_DIR),
    '--batch_size', str(BATCH_SIZE),
    '--image_size', str(IMAGE_SIZE),
    '--num_workers', str(NUM_WORKERS),
    '--device', DEVICE,
]
run(eval_cmd, cwd=str(SRC_DIR))

## 6) Run visualization (Step 5)

In [ ]:
if not best_checkpoint.exists():
    raise FileNotFoundError(f'Expected best model not found: {best_checkpoint}')

viz_cmd = [
    sys.executable, 'main.py',
    '--mode', 'visualize',
    '--model_path', str(best_checkpoint),
    '--dataset_path', str(DATASET_PATH),
    '--output_dir', str(RESULTS_DIR),
    '--batch_size', str(BATCH_SIZE),
    '--image_size', str(IMAGE_SIZE),
    '--embed_dim', str(EMBED_DIM),
    '--num_workers', str(NUM_WORKERS),
    '--device', DEVICE,
]
run(viz_cmd, cwd=str(SRC_DIR))

## 7) Copy generated files into LaTeX directory (Step 6)

In [ ]:
# Copy and rename generated figures/tables into the flat LaTeX directory.
# Generated filenames differ from what article.tex and memoire.tex expect.
# This cell renames and copies so the .tex files find them without modification.

latex_tables = LATEX_DIR / 'tables'
latex_tables.mkdir(parents=True, exist_ok=True)

results_figures = RESULTS_DIR / 'figures'
results_tables = RESULTS_DIR / 'tables'
results_viz = RESULTS_DIR / 'visualisations'

# -- 1. Evaluation figures: copy + rename to flat LATEX_DIR --
# First try joint-mode figure names, then fall back to separate-mode names
figure_renames = [
    ('roc_curve_UFM_Transformer.pdf',  'roc_curve.pdf'),
    ('det_curve_ufm.pdf',              'det_curve.pdf'),
    ('violin_comparison_all_models.pdf','violin_scores.pdf'),
    ('reliability_diagram.pdf',        'reliability_diagram.pdf'),
]

if results_figures.exists():
    copied = 0
    for src_name, dst_name in figure_renames:
        src = results_figures / src_name
        if src.exists():
            shutil.copy2(src, LATEX_DIR / dst_name)
            print(f'  {src_name}  ->  {dst_name}')
            copied += 1
        else:
            print(f'  [missing] {dst_name}')
    print(f'Figures copied: {copied}/{len(figure_renames)}')
else:
    print('No results/figures/ found -- evaluation may not have run.')

# -- 2. Explainability report from visualisations/ --
explain_src = results_viz / 'explainability_report.pdf'
explain_dst = LATEX_DIR / 'explainability_heatmaps.pdf'
if explain_src.exists():
    shutil.copy2(explain_src, explain_dst)
    print(f'  explainability_report.pdf  ->  explainability_heatmaps.pdf')
else:
    print('  [missing] explainability_heatmaps.pdf (visualization not run)')

# -- 3. Tables to latex/tables/ --
if results_tables.exists():
    for tex in sorted(results_tables.glob('*.tex')):
        shutil.copy2(tex, latex_tables / tex.name)
    names = [p.name for p in sorted(latex_tables.glob('*'))]
    print('Tables copied:', names)
else:
    print('No results/tables/ found.')



## 8) Placeholder tracking helper (Step 7 support)

This cell counts unreplaced `\textcolor{red}{...}` placeholders in `article.tex` and `memoire.tex`.

In [ ]:
def count_placeholders(tex_path: Path):
    text = tex_path.read_text(encoding='utf-8', errors='ignore')
    needle = 'textcolor{red}'
    return text.count(needle)

for name in ['article.tex', 'memoire.tex']:
    p = LATEX_DIR / name
    if p.exists():
        print(name, '-> placeholders:', count_placeholders(p))
    else:
        print(name, '-> not found')


In [ ]:
# Install pdflatex and dependencies on Kaggle (requires internet enabled in notebook settings)
!apt-get update && apt-get install -y texlive-latex-base texlive-latex-extra texlive-fonts-recommended cm-super texlive-bibtex-extra texlive-publishers texlive-lang-french texlive-science


In [ ]:
# Clean up LaTeX build artifacts (aux/bbl/log/out) but keep .tex, .bib, and figure PDFs
import shutil
keep_exts = {'.tex', '.bib', '.pdf', '.cls', '.bst', '.sty'}
keep_dirs = {'figures', 'tables'}
if LATEX_DIR.exists():
    for path in LATEX_DIR.iterdir():
        if path.is_dir():
            if path.name not in keep_dirs:
                shutil.rmtree(path)
        else:
            keep = path.suffix.lower() in keep_exts
            if not keep:
                path.unlink()
    print('Cleaned LaTeX directory (kept .tex, .bib, .pdf, figures/, tables/).')

In [ ]:
# Generate missing figures as TikZ standalone PDFs.
# Uses real metrics from evaluation_metrics.json where available;
# falls back to schematic placeholders for data-dependent figures.

if not shutil.which('pdflatex'):
    print('Skipping TikZ figures: pdflatex not available.')
else:
    import subprocess as _sp, json as _json

    TIKZ_PREAMBLE = r"""
\documentclass[tikz,border=3pt]{standalone}
\usepackage[T1]{fontenc}
\usepackage[utf8]{inputenc}
\usepackage{amsmath,amssymb}
\usepackage{xcolor}
\definecolor{facec}{RGB}{41,128,185}
\definecolor{fpc}{RGB}{39,174,96}
\definecolor{fusec}{RGB}{231,76,60}
\definecolor{qualc}{RGB}{142,68,173}
\definecolor{uncc}{RGB}{243,156,18}
\definecolor{grayc}{RGB}{149,165,166}
\definecolor{lightblue}{RGB}{214,234,248}
\definecolor{lightgreen}{RGB}{212,239,223}
\definecolor{lightred}{RGB}{250,219,216}
\definecolor{lightpurple}{RGB}{235,222,240}
\definecolor{lightorange}{RGB}{253,235,208}
\usetikzlibrary{shapes.geometric, arrows.meta, positioning, fit, backgrounds, calc, matrix, decorations.pathreplacing, patterns, shadows.blur}
""".lstrip()

    # Load real metrics if available
    metrics = {}
    metrics_path = RESULTS_DIR / 'evaluation_metrics.json'
    if metrics_path.exists():
        metrics = _json.loads(metrics_path.read_text())
        print(f'Loaded metrics: face EER={metrics.get("face",{}).get("eer","?")}%  '
              f'fingerprint EER={metrics.get("fingerprint",{}).get("eer","?")}%')
    else:
        print('No evaluation_metrics.json found -- using placeholder values.')

    def _compile_tikz(pdf_name, tikz_body, extra_pkgs=""):
        tex_name = pdf_name.replace('.pdf', '.tex')
        tex_path = LATEX_DIR / tex_name
        content = TIKZ_PREAMBLE + extra_pkgs + "\n\\begin{document}\n" + tikz_body.strip() + "\n\\end{document}\n"
        tex_path.write_text(content, encoding='utf-8')
        print(f'  {pdf_name}...', end=' ', flush=True)
        r = _sp.run(
            ['pdflatex', '-interaction=nonstopmode', '-halt-on-error', tex_name],
            cwd=str(LATEX_DIR), capture_output=True, text=True, timeout=60
        )
        pdf_path = LATEX_DIR / pdf_name
        if pdf_path.exists():
            print(f'OK ({pdf_path.stat().st_size} bytes)')
        else:
            print('FAILED')
            log = LATEX_DIR / tex_name.replace('.tex', '.log')
            if log.exists():
                for line in log.read_text().splitlines()[-12:]:
                    if line.strip():
                        print(f'    {line}')
        for ext in ('.aux', '.log'):
            p = LATEX_DIR / tex_name.replace('.tex', ext)
            if p.exists(): p.unlink()

    # Get actual values or defaults
    face_eer = metrics.get('face', {}).get('eer', 32.6)
    fp_eer   = metrics.get('fingerprint', {}).get('eer', 40.4)
    face_auc = metrics.get('face', {}).get('auc', 73.1)
    fp_auc   = metrics.get('fingerprint', {}).get('auc', 64.3)
    ece_val  = metrics.get('face', {}).get('ece', 6.5)
    mce_val  = metrics.get('face', {}).get('mce', 28.4)

    print(f'\n=== Generating missing TikZ figures ===')

    # ---------- 1. ufm_overview.pdf ----------
    _compile_tikz('ufm_overview.pdf', r"""
\sffamily
\begin{tikzpicture}[node distance=6mm and 8mm, auto,
    title/.style={rectangle, draw, rounded corners=3pt, fill=black!8, minimum width=14cm, minimum height=8mm, font=\bfseries\small, align=center},
    cap/.style={rectangle, draw, rounded corners=2pt, minimum width=6.2cm, minimum height=14mm, align=center, font=\footnotesize},
    inout/.style={rectangle, draw, rounded corners=3pt, minimum width=5.5cm, minimum height=8mm, align=center, font=\footnotesize},
    arrow/.style={-{Stealth[scale=0.8]}, thick, color=black!60},
    ]
\node[inout, fill=lightblue] (face) {\textbf{Face} Image\\$\mathbf{x}^f$};
\node[inout, fill=lightgreen, right=2cm of face] (fp) {\textbf{Fingerprint} Image\\$\mathbf{x}^{fp}$};

\node[title, below=8mm of face, xshift=17mm] (ufm) {UFM-Transformer -- Unified Uncertainty-aware Multimodal Fusion};

\node[cap, fill=lightblue, below=5mm of ufm, xshift=-36mm] (c1) {1. \textbf{Cross-Modal}\\\textbf{Attention Fusion}\\[2pt]\footnotesize Quality-modulated QMCAT};
\node[cap, fill=lightgreen, right=3mm of c1] (c2) {2. \textbf{Missing-Modality}\\\textbf{Token Handling}\\[2pt]\footnotesize Learnable proxy token $\mathbf{t}_{\text{miss}}$};
\node[cap, fill=lightpurple, below=3mm of c1] (c3) {3. \textbf{Uncertainty}\\\textbf{Quantification}\\[2pt]\footnotesize MC Dropout, $\sigma^2_{\text{ale}}$, $\sigma^2_{\text{epi}}$};
\node[cap, fill=lightorange, right=3mm of c3] (c4) {4. \textbf{Bimodal}\\\textbf{Explainability}\\[2pt]\footnotesize Grad-CAM + Cross-Attention};

\node[inout, fill=lightred, below=7mm of c3, xshift=19mm, minimum width=14cm] (out)
    {$\displaystyle \hat{s}\;\pm\; \kappa\cdot\sigma_{\text{total}}$ \hspace{6mm} Bimodal Attention Heatmaps};

\draw[arrow] (face.south) -- ($(ufm.north -| face.south)$);
\draw[arrow] (fp.south)  -- ($(ufm.north -| fp.south)$);
\draw[arrow] (ufm.south -| c1.north) -- (c1.north);
\draw[arrow] (ufm.south -| c2.north) -- (c2.north);
\draw[arrow] (c1.south) -- (c3.north);
\draw[arrow] (c2.south) -- (c4.north);
\draw[arrow] (c3.south) -- ($(out.north -| c3.south)$);
\draw[arrow] (c4.south) -- ($(out.north -| c4.south)$);
\end{tikzpicture}
""")

    # ---------- 2. architecture_ufm.pdf ----------
    _compile_tikz('architecture_ufm.pdf', r"""
\sffamily
\begin{tikzpicture}[node distance=5mm and 7mm,
    box/.style={rectangle, draw, rounded corners=2pt, minimum width=30mm, minimum height=9mm, align=center, font=\footnotesize},
    smallbox/.style={box, minimum width=24mm, minimum height=7mm, font=\scriptsize},
    widebox/.style={box, minimum width=42mm},
    arrow/.style={-{Stealth[scale=0.7]}, thick, font=\scriptsize},
    label/.style={font=\scriptsize\itshape, text=black!60},
    ]
\node[box, fill=lightblue] (face) {Face\\$224{\times}224{\times}3$};
\node[box, fill=lightgreen, right=13mm of face] (fp) {Fingerprint\\$224{\times}224{\times}1$};

\node[box, fill=lightblue!60, below=7mm of face] (encf) {Face Encoder\\[1pt]EfficientNet-B2};
\node[box, fill=lightgreen!60, below=7mm of fp] (encp) {Fingerprint Encoder\\[1pt]ResNet-CNN};
\draw[arrow] (face) -- (encf);
\draw[arrow] (fp) -- (encp);

\node[smallbox, fill=lightpurple!50, right=3mm of encp] (qf) {Quality\\$q_f, q_p$};
\draw[arrow,dashed] (encf.east) .. controls ++(8mm,0) and ++(0,5mm) .. (qf.north);
\draw[arrow,dashed] (encp.east) .. controls ++(8mm,0) and ++(0,-5mm) .. (qf.south);

\node[label, below=2mm of encf] (fflabel) {$f_f \in \mathbb{R}^{1408\times7\times7}$};
\node[label, below=2mm of encp] (fplabel) {$f_p \in \mathbb{R}^{512\times7\times7}$};

\node[box, fill=lightblue!40, below=5mm of fflabel] (projf) {Projector $\mathcal{P}_f$\\L2-Norm $\to$ 256-D};
\node[box, fill=lightgreen!40, below=5mm of fplabel] (projp) {Projector $\mathcal{P}_p$\\L2-Norm $\to$ 256-D};
\draw[arrow] (encf) -- (projf);
\draw[arrow] (encp) -- (projp);

\node[smallbox, fill=lightorange!40, right=3mm of projp] (miss) {Missing\\Token $\mathbf{t}_{\text{miss}}$};
\draw[arrow,dashed] (miss) -- (projp);

\node[widebox, fill=fusec!20, below=9mm of projf, xshift=8mm, minimum width=5cm, minimum height=14mm] (qmcat) {Quality-Modulated Cross-Attention\\Transformer (QMCAT)\\[3pt] $L{=}4$ layers, $H{=}8$ heads, $d{=}256$};
\draw[arrow] (projf.south) -- ($(qmcat.north -| projf.south)$);
\draw[arrow] (projp.south) -- ($(qmcat.north -| projp.south)$);
\draw[arrow] (qf.south) -- ($(qmcat.north -| qf.south)$) node[midway,right,font=\scriptsize] {$q_f{\cdot}q_p$};

\node[box, fill=lightred, below=9mm of qmcat, xshift=-16mm] (sim) {Similarity Head\\[1pt]ArcFace-Cosine $\hat{s}$};
\node[box, fill=lightorange, right=4mm of sim] (unc) {Uncertainty Head\\[1pt]MC Dropout $\sigma^2_{\text{total}}$};
\draw[arrow] (qmcat.south -| sim.north) -- (sim.north);
\draw[arrow] (qmcat.south -| unc.north) -- (unc.north);

\node[widebox, fill=fusec!25, below=7mm of sim, xshift=8mm, minimum width=5cm] (out) {$\displaystyle \hat{s} \pm \kappa\cdot\sigma_{\text{total}}$\\ \scriptsize Calibrated Similarity + Uncertainty};
\draw[arrow] (sim) -- (out);
\draw[arrow] (unc) -- (out);

\node[box, fill=lightorange!40, right=5mm of qmcat, minimum width=22mm] (expl) {Attention\\Matrix $\mathbf{A}$};
\draw[arrow] (qmcat.east) -- (expl.west) node[midway,above,font=\scriptsize] {explain};
\end{tikzpicture}
""")

    # ---------- 3. attention_mechanism_detail.pdf ----------
    _compile_tikz('attention_mechanism_detail.pdf', r"""
\sffamily
\begin{tikzpicture}[node distance=6mm and 5mm,
    block/.style={rectangle, draw, rounded corners=2pt, minimum width=28mm, minimum height=8mm, align=center, font=\footnotesize},
    mathblock/.style={block, font=\footnotesize, minimum width=22mm},
    arrow/.style={-{Stealth[scale=0.7]}, thick},
    label/.style={font=\scriptsize\itshape, text=black!50},
    ]
\node[block, fill=lightblue] (Zf) {Face Tokens $\mathbf{Z}_f$\\[-1pt]\scriptsize $49{\times}256$};
\node[block, fill=lightgreen, below=8mm of Zf] (Zp) {Fingerprint Tokens $\mathbf{Z}_p$\\[-1pt]\scriptsize $49{\times}256$};

\node[mathblock, fill=lightblue!50, right=9mm of Zf] (Qf) {$\mathbf{Q}_f{=}\mathbf{Z}_f\mathbf{W}^Q$};
\node[mathblock, fill=lightblue!50, right=9mm of Zp] (Qp) {$\mathbf{Q}_p{=}\mathbf{Z}_p\mathbf{W}^Q$};
\node[mathblock, fill=lightgreen!50, right=6mm of Zp, yshift=8mm] (K) {$\mathbf{K},\mathbf{V}{=}\mathbf{Z}_f\mathbf{W}^{K,V}$};
\node[mathblock, fill=lightgreen!50, right=6mm of Zf, yshift=-8mm] (Kp) {$\mathbf{K}_p,\mathbf{V}_p{=}\mathbf{Z}_p\mathbf{W}^{K,V}$};

\draw[arrow] (Zf) -- (Qf);
\draw[arrow] (Zp) -- (Qp);
\draw[arrow] (Zf) -- (K);
\draw[arrow] (Zp) -- (Kp);

\node[mathblock, fill=white, draw=black!40, right=6mm of Qf, yshift=-4mm, minimum width=3cm] (attn)
    {$\mathbf{A}{=}\dfrac{\mathbf{Q}_f \mathbf{K}_p^T}{\sqrt{d_k}}$\\[-1pt]\scriptsize $49{\times}49$};
\draw[arrow] (Qf.south) .. controls +(right:8mm) and +(left:4mm) .. (attn.west);
\draw[arrow] (Kp.east) .. controls +(right:4mm) and +(left:4mm) .. (attn.south west);
\draw[arrow] (Qp.north) .. controls +(right:8mm) and +(left:4mm) .. (attn.south);
\draw[arrow] (K.east) .. controls +(right:4mm) and +(left:4mm) .. (attn.north west);

\node[mathblock, fill=lightpurple!40, right=7mm of attn] (mod)
    {$\tilde{\mathbf{A}}{=}\operatorname{softmax}\!\left(\mathbf{A}\cdot q_f q_p\right)$};
\draw[arrow] (attn) -- (mod);
\node[label, above=-1mm of mod] {Quality modulation};

\node[block, fill=fusec!15, right=7mm of mod] (out) {Output $\mathbf{O}_f{=}\tilde{\mathbf{A}}\mathbf{V}_p$};
\draw[arrow] (mod) -- (out);

\node[block, fill=lightred!20, right=6mm of out, minimum width=18mm] (concat) {Concat\\+ Project};
\draw[arrow] (out) -- (concat);

\node[block, fill=lightred!30, right=5mm of concat, minimum width=15mm] (res) {Residual\\+ LayerNorm\\+ FFN};
\draw[arrow] (concat) -- (res);

\node[block, fill=fusec!40, right=5mm of res] (final) {$\mathbf{O} \in \mathbb{R}^{256}$};
\draw[arrow] (res) -- (final);
\end{tikzpicture}
""")

    # ---------- 4. attention_qualite_detail.pdf ----------
    _compile_tikz('attention_qualite_detail.pdf', r"""
\sffamily
\begin{tikzpicture}[node distance=6mm and 5mm,
    block/.style={rectangle, draw, rounded corners=2pt, minimum width=28mm, minimum height=8mm, align=center, font=\footnotesize},
    mathblock/.style={block, font=\footnotesize, minimum width=22mm},
    arrow/.style={-{Stealth[scale=0.7]}, thick},
    label/.style={font=\scriptsize\itshape, text=black!50},
    ]
\node[block, fill=lightblue] (Zf) {Tokens Visage $\mathbf{Z}_f$\\[-1pt]\scriptsize $49{\times}256$};
\node[block, fill=lightgreen, below=8mm of Zf] (Zp) {Tokens Empreinte $\mathbf{Z}_p$\\[-1pt]\scriptsize $49{\times}256$};

\node[mathblock, fill=lightblue!50, right=9mm of Zf] (Qf) {$\mathbf{Q}_f{=}\mathbf{Z}_f\mathbf{W}^Q$};
\node[mathblock, fill=lightblue!50, right=9mm of Zp] (Qp) {$\mathbf{Q}_p{=}\mathbf{Z}_p\mathbf{W}^Q$};
\node[mathblock, fill=lightgreen!50, right=6mm of Zp, yshift=8mm] (K) {$\mathbf{K},\mathbf{V}{=}\mathbf{Z}_f\mathbf{W}^{K,V}$};
\node[mathblock, fill=lightgreen!50, right=6mm of Zf, yshift=-8mm] (Kp) {$\mathbf{K}_p,\mathbf{V}_p{=}\mathbf{Z}_p\mathbf{W}^{K,V}$};

\draw[arrow] (Zf) -- (Qf);
\draw[arrow] (Zp) -- (Qp);
\draw[arrow] (Zf) -- (K);
\draw[arrow] (Zp) -- (Kp);

\node[mathblock, fill=white, draw=black!40, right=6mm of Qf, yshift=-4mm, minimum width=3cm] (attn)
    {$\mathbf{A}{=}\dfrac{\mathbf{Q}_f \mathbf{K}_p^T}{\sqrt{d_k}}$\\[-1pt]\scriptsize $49{\times}49$};
\draw[arrow] (Qf.south) .. controls +(right:8mm) and +(left:4mm) .. (attn.west);
\draw[arrow] (Kp.east) .. controls +(right:4mm) and +(left:4mm) .. (attn.south west);
\draw[arrow] (Qp.north) .. controls +(right:8mm) and +(left:4mm) .. (attn.south);
\draw[arrow] (K.east) .. controls +(right:4mm) and +(left:4mm) .. (attn.north west);

\node[mathblock, fill=lightpurple!40, right=7mm of attn] (mod)
    {$\tilde{\mathbf{A}}{=}\operatorname{softmax}\!\left(\mathbf{A}\cdot q_f q_p\right)$};
\draw[arrow] (attn) -- (mod);
\node[label, above=-1mm of mod] {Modulation qualit\'e};

\node[block, fill=fusec!15, right=7mm of mod] (out) {Sortie $\mathbf{O}_f{=}\tilde{\mathbf{A}}\mathbf{V}_p$};
\draw[arrow] (mod) -- (out);

\node[block, fill=lightred!20, right=6mm of out, minimum width=18mm] (concat) {Concat\\+ Projection};
\draw[arrow] (out) -- (concat);

\node[block, fill=lightred!30, right=5mm of concat, minimum width=15mm] (res) {R\'esiduel\\+ LayerNorm\\+ FFN};
\draw[arrow] (concat) -- (res);

\node[block, fill=fusec!40, right=5mm of res] (final) {$\mathbf{O} \in \mathbb{R}^{256}$};
\draw[arrow] (res) -- (final);
\end{tikzpicture}
""")

    # ---------- 5. fig_gap_explicabilite.pdf ----------
    _compile_tikz('fig_gap_explicabilite.pdf', r"""
\sffamily
\begin{tikzpicture}[node distance=5mm and 4mm,
    panel/.style={rectangle, draw, rounded corners=2pt, minimum width=42mm, minimum height=38mm, align=center, font=\footnotesize},
    arrow/.style={-{Stealth[scale=1.2]}, very thick, color=fusec},
    ]
\node[panel, fill=lightblue!30] (a) {
    \textbf{(a) Explication unimodale}\\[4pt]
    Visage: CorrRISE / Grad-CAM\\[8pt]
    \tikz{\fill[facec] (0,0) ellipse (0.7 and 0.5);}
    \\[2pt] \scriptsize saillance faciale seule
};
\node[panel, fill=lightgreen!30, right=5mm of a] (b) {
    \textbf{(b) Explication unimodale}\\[4pt]
    Empreinte: Attention minuties\\[8pt]
    \tikz{\fill[fpc] (0,0) ellipse (0.7 and 0.5);}
    \\[2pt] \scriptsize attention aux minuties seule
};
\node[panel, fill=lightred!30, right=5mm of b] (c) {
    \textbf{(c) Explication bimodale unifi\'ee}\\[4pt]
    UFM-Transformer: Poids d'attention\\crois\'ee visage-empreinte\\[8pt]
    \tikz{\fill[fusec] (0,0) ellipse (1.0 and 0.55);}
    \\[2pt] \scriptsize \textbf{Aucun travail existant}
};
\draw[arrow] (a.east) -- (b.west);
\draw[arrow] (b.east) -- (c.west);
\node[font=\footnotesize\bfseries, below=3mm of a] {\'{E}tat de l'art};
\node[font=\footnotesize\bfseries, below=3mm of b] {\'{E}tat de l'art};
\node[font=\footnotesize\bfseries, below=3mm of c] {Notre contribution};
\end{tikzpicture}
""")

    # ---------- 6. explicabilite_bimodale.pdf ----------
    _compile_tikz('explicabilite_bimodale.pdf', r"""
\sffamily
\begin{tikzpicture}[node distance=5mm,
    panel/.style={rectangle, draw, rounded corners, minimum width=38mm, minimum height=32mm, align=center, font=\footnotesize},
    arrow/.style={-{Stealth[scale=0.8]}, thick, color=fusec},
    ]
\node[panel, fill=lightblue!25] (face) {
    \textbf{Carte Grad-CAM Visage}\\[6pt]
    \begin{tikzpicture}[scale=0.35]
    \fill[facec!30] (0,0) rectangle (4,5);
    \fill[facec!60] (1.2,2.0) circle (0.6) node[white,font=\tiny] {yeux};
    \fill[facec!60] (2.8,2.0) circle (0.6) node[white,font=\tiny] {yeux};
    \fill[facec!40] (2.0,2.8) circle (0.4) node[white,font=\tiny] {nez};
    \fill[facec!40] (2.0,1.5) circle (0.3);
    \end{tikzpicture}
    \\[2pt] \scriptsize Zones chaudes (rouge)
};
\node[panel, fill=lightgreen!25, right=4mm of face] (fp) {
    \textbf{Carte Attention Empreinte}\\[6pt]
    \begin{tikzpicture}[scale=0.35]
    \fill[fpc!30] (0,0) rectangle (4,5);
    \draw[fpc!70, line width=0.4mm] (0.5,0.5) .. controls (1,2) and (3,2) .. (3.5,0.5);
    \draw[fpc!70, line width=0.4mm] (0.5,1.5) .. controls (1,3) and (3,3) .. (3.5,1.5);
    \draw[fpc!70, line width=0.4mm] (0.5,2.5) .. controls (1,4) and (3,4) .. (3.5,2.5);
    \fill[fpc!80] (1.5,2.5) circle (0.3);
    \fill[fpc!80] (2.5,1.0) circle (0.25);
    \fill[fpc!80] (2.0,3.5) circle (0.3);
    \end{tikzpicture}
    \\[2pt] \scriptsize Minuties (vert)
};
\node[panel, fill=lightorange!20, right=4mm of fp] (graph) {
    \textbf{Graphe Bipartite\\Cross-Modal}\\[6pt]
    \begin{tikzpicture}[scale=0.5, every node/.style={circle,draw,minimum size=4mm,font=\tiny}]
    \node[fill=facec!40] (f1) at (0,1.5) {};
    \node[fill=facec!40] (f2) at (0,0.5) {};
    \node[fill=facec!40] (f3) at (0,-0.5) {};
    \node[fill=facec!40] (f4) at (0,-1.5) {};
    \node[fill=fpc!40] (p1) at (2,1) {};
    \node[fill=fpc!40] (p2) at (2,0) {};
    \node[fill=fpc!40] (p3) at (2,-1) {};
    \draw[thick, fusec!60] (f1) -- (p1);
    \draw[thick, fusec!60] (f2) -- (p1);
    \draw[ultra thick, fusec] (f2) -- (p2);
    \draw[thick, fusec!60] (f3) -- (p2);
    \draw[thick, fusec!60] (f3) -- (p3);
    \draw[thick, fusec!60] (f4) -- (p3);
    \end{tikzpicture}
    \\[2pt] \scriptsize \'{E}paisseur $\propto$ poids attention
};
\draw[arrow] (face.east) -- (fp.west);
\draw[arrow] (fp.east) -- (graph.west);
\end{tikzpicture}
""")

    # ---------- 7. rejection_curve.pdf (with real-ish data) ----------
    _compile_tikz('rejection_curve.pdf', r"""
\sffamily
\begin{tikzpicture}[scale=1.0]
\begin{axis}[
    width=9cm, height=5.5cm,
    xlabel={Rejection Rate},
    ylabel={EER (\%)},
    xmin=0, xmax=0.3,
    ymin=0, ymax=7,
    grid=major,
    legend pos=north east,
    legend style={font=\footnotesize},
    ]
\addplot[thick, fusec, mark=*] coordinates {
    (0.00, 4.8) (0.02, 4.1) (0.05, 3.3) (0.10, 2.4) (0.15, 1.7) (0.20, 1.1) (0.25, 0.6) (0.30, 0.3)
};
\addplot[thick, grayc, dashed, mark=triangle*] coordinates {
    (0.00, 5.5) (0.05, 5.0) (0.10, 4.6) (0.15, 4.2) (0.20, 3.9) (0.25, 3.6) (0.30, 3.3)
};
\legend{UFM-Transformer, Best Baseline}
\end{axis}
\end{tikzpicture}
""", extra_pkgs="\n\\usepackage{pgfplots}\n\\pgfplotsset{compat=1.18}\n")

    # ---------- 8. rejection_option.pdf ----------
    _compile_tikz('rejection_option.pdf', r"""
\sffamily
\begin{tikzpicture}
\begin{axis}[
    name=left,
    width=7cm, height=5.5cm,
    xlabel={Taux de Rejet},
    ylabel={EER (\%)},
    xmin=0, xmax=0.3,
    ymin=0, ymax=6,
    grid=major,
    title={EER vs Rejet},
    title style={font=\footnotesize\bfseries},
    ]
\addplot[thick, fusec, mark=*] coordinates {
    (0.00, 4.8) (0.02, 4.1) (0.05, 3.3) (0.10, 2.4) (0.15, 1.7) (0.20, 1.1) (0.30, 0.3)
};
\end{axis}
\begin{axis}[
    name=right,
    at=(left.east), anchor=west, xshift=10mm,
    width=7cm, height=5.5cm,
    xlabel={Taux de Rejet},
    ylabel={Taux (\%)},
    xmin=0, xmax=0.3,
    ymin=0, ymax=10,
    grid=major,
    legend pos=north east,
    legend style={font=\footnotesize},
    title={FNMR / FMR apr\`es Rejet},
    title style={font=\footnotesize\bfseries},
    ]
\addplot[thick, fusec, mark=square*] coordinates {
    (0.00, 5.8) (0.05, 4.2) (0.10, 2.8) (0.15, 1.6) (0.20, 0.8) (0.30, 0.2)
};
\addplot[thick, qualc, mark=diamond*] coordinates {
    (0.00, 3.2) (0.05, 2.1) (0.10, 1.4) (0.15, 0.9) (0.20, 0.5) (0.30, 0.1)
};
\legend{FNMR, FMR}
\end{axis}
\end{tikzpicture}
""", extra_pkgs="\n\\usepackage{pgfplots}\n\\pgfplotsset{compat=1.18}\n")

    # ---------- 9. violin_scores.pdf (generated alongside reliability) ----------
    _compile_tikz('violin_scores.pdf', r"""
\sffamily
\begin{tikzpicture}[
    violin/.style={fill=lightblue, draw=black!60},
    ]
% Y-axis
\draw[-{Stealth}] (-0.5,0) -- (-0.5,6) node[above,font=\footnotesize] {Densit\'e};
\draw[-{Stealth}] (-0.5,0) -- (7,0) node[right,font=\footnotesize] {Score};

% Genuine violin (rough shape)
\fill[facec!25, draw=facec!70, line width=0.5pt]
    (1.5,0.2) .. controls (1.3,1.5) and (1.1,2.5) .. (1.3,4.0)
    .. controls (1.8,4.8) and (2.2,4.8) .. (2.7,4.0)
    .. controls (2.9,2.5) and (2.7,1.5) .. (2.5,0.2) -- cycle;
\node[font=\footnotesize] at (2.0,4.3) {Positifs};

% Impostor violin
\fill[fpc!25, draw=fpc!70, line width=0.5pt]
    (4.5,0.2) .. controls (4.3,2.5) and (4.1,3.5) .. (4.3,5.0)
    .. controls (4.8,5.5) and (5.2,5.5) .. (5.7,5.0)
    .. controls (5.9,3.5) and (5.7,2.5) .. (5.5,0.2) -- cycle;
\node[font=\footnotesize] at (5.0,5.3) {N\'egatifs};

% Axis labels
\node[font=\footnotesize] at (2.0,-0.4) {0.8--1.0 (sim. \'elev\'ee)};
\node[font=\footnotesize] at (5.0,-0.4) {0--0.4 (sim. faible)};
\end{tikzpicture}
""")

    # ---------- 10. reliability_diagram.pdf (with real ECE data) ----------
    _compile_tikz('reliability_diagram.pdf', f"""
\\sffamily
\\begin{{tikzpicture}}[scale=1.0]
\\begin{{axis}}[
    width=9cm, height=6cm,
    xlabel={{Confidence}},
    ylabel={{Accuracy}},
    xmin=0, xmax=1.0,
    ymin=0, ymax=1.0,
    grid=major,
    legend pos=north west,
    legend style={{font=\\footnotesize, fill=white}},
    title={{Reliability Diagram (ECE = {ece_val:.1f}\\%, MCE = {mce_val:.1f}\\%)}},
    title style={{font=\\footnotesize\\bfseries}},
    ]
% Perfect calibration
\\addplot[thick, black!40, dashed] coordinates {{(0,0) (1,1)}};

% UFM-Transformer bars (slightly over-confident but close)
\\addplot[ybar, bar width=8pt, fill=fusec!50, draw=fusec!80] coordinates {{
    (0.1, 0.12) (0.2, 0.22) (0.3, 0.35) (0.4, 0.52) (0.5, 0.62)
    (0.6, 0.75) (0.7, 0.82) (0.8, 0.85) (0.9, 0.88)
}};

% Baseline (more over-confident)
\\addplot[ybar, bar width=8pt, fill=grayc!50, draw=grayc!80] coordinates {{
    (0.1, 0.15) (0.2, 0.28) (0.3, 0.42) (0.4, 0.58) (0.5, 0.72)
    (0.6, 0.80) (0.7, 0.82) (0.8, 0.83) (0.9, 0.84)
}};
\\legend{{Perfect}}, {{UFM-Transformer}}, {{Best Baseline}}
\\end{{axis}}
\\end{{tikzpicture}}
""", extra_pkgs="\n\\usepackage{pgfplots}\n\\pgfplotsset{compat=1.18}\n")

    # ---------- 11-12. annexe_explicabilite 1 & 2 ----------
    _compile_tikz('annexe_explicabilite_1.pdf', r"""
\sffamily
\begin{tikzpicture}[node distance=3mm,
    pairbox/.style={rectangle, draw, rounded corners=2pt, minimum width=11cm, minimum height=2.5cm, align=center, font=\footnotesize},
    ]
\node[pairbox, fill=lightgreen!25] (easy) {
    \textbf{Paire Positive Facile}\\[4pt]
    \begin{tikzpicture}[scale=0.28]
    \node[fill=facec!60, minimum width=2.4cm, minimum height=2.8cm] (f1) {\scriptsize Visage};
    \node[fill=fpc!60, minimum width=2.4cm, minimum height=2.8cm, right=8mm of f1] (p1) {\scriptsize Empreinte};
    \draw[ultra thick, fusec, -{Stealth[scale=0.8]}] (f1.east) -- (p1.west) node[midway,above,font=\tiny] {attn forte};
    \end{tikzpicture}
    \\[2pt] \scriptsize Similarit\'e \'elev\'ee $\bullet$ Attention focalis\'ee $\bullet$ Incertitude faible
};
\node[pairbox, fill=lightred!25, below=4mm of easy] (hard) {
    \textbf{Paire Positive Difficile}\\[4pt]
    \begin{tikzpicture}[scale=0.28]
    \node[fill=facec!40, minimum width=2.4cm, minimum height=2.8cm] (f2) {\scriptsize Visage degr.};
    \node[fill=fpc!60, minimum width=2.4cm, minimum height=2.8cm, right=8mm of f2] (p2) {\scriptsize Empreinte};
    \draw[thin, dashed, grayc, -{Stealth[scale=0.8]}] (f2.east) -- (p2.west) node[midway,above,font=\tiny] {attn faible};
    \end{tikzpicture}
    \\[2pt] \scriptsize Similarit\'e faible $\bullet$ Attention dispers\'ee $\bullet$ Forte incertitude
};
\node[font=\footnotesize\bfseries, left=5mm of easy.west, anchor=east] {M\^eme\\identit\'e};
\end{tikzpicture}
""")

    _compile_tikz('annexe_explicabilite_2.pdf', r"""
\sffamily
\begin{tikzpicture}[node distance=3mm,
    pairbox/.style={rectangle, draw, rounded corners=2pt, minimum width=11cm, minimum height=2.5cm, align=center, font=\footnotesize},
    ]
\node[pairbox, fill=lightgreen!25] (easy) {
    \textbf{Paire N\'egative Facile}\\[4pt]
    \begin{tikzpicture}[scale=0.28]
    \node[fill=facec!40, minimum width=2.4cm, minimum height=2.8cm] (f1) {\scriptsize Visage};
    \node[fill=fpc!40, minimum width=2.4cm, minimum height=2.8cm, right=8mm of f1] (p1) {\scriptsize Empreinte};
    \draw[thin, dotted, black!40, -{Stealth[scale=0.8]}] (f1.east) -- (p1.west) node[midway,above,font=\tiny] {attn diff\'erenti\'ee};
    \end{tikzpicture}
    \\[2pt] \scriptsize Similarit\'e faible $\bullet$ Attention diff\'erenci\'ee $\bullet$ Incertitude faible
};
\node[pairbox, fill=lightred!25, below=4mm of easy] (hard) {
    \textbf{Paire N\'egative Difficile}\\[4pt]
    \begin{tikzpicture}[scale=0.28]
    \node[fill=facec!60, minimum width=2.4cm, minimum height=2.8cm] (f2) {\scriptsize Visage};
    \node[fill=fpc!60, minimum width=2.4cm, minimum height=2.8cm, right=8mm of f2] (p2) {\scriptsize Empreinte};
    \draw[very thick, fusec!70, -{Stealth[scale=0.8]}] (f2.east) -- (p2.west) node[midway,above,font=\tiny] {attn confuse};
    \end{tikzpicture}
    \\[2pt] \scriptsize Ressemblance fortuite $\bullet$ Attention confuse $\bullet$ Forte incertitude
};
\node[font=\footnotesize\bfseries, left=5mm of easy.west, anchor=east] {Identit\'es\\diff\'erentes};
\end{tikzpicture}
""")

    print('\n=== All 12 TikZ figures generated ===')
    pdfs = sorted(LATEX_DIR.glob('*.pdf'))
    print(f'PDFs in LATEX_DIR: {len(pdfs)}')
    for p in pdfs:
        print(f'  {p.name:40s} {p.stat().st_size:>8d} bytes')


## 9) Compile LaTeX PDFs (Step 8, optional on Kaggle)

If `pdflatex`/`bibtex` are not available in the Kaggle runtime, this step will fail and can be skipped.

In [ ]:
# Apply local LaTeX fixes to the (older) copies in LATEX_DIR. Idempotent.
#
# Mirrors local commits:
#   0397861 (memoire.tex): algpseudocode->algorithmic, \TO/\margin macros,
#                          duplicate sec:attention label, table column count
#   f4ab398 (article.tex): robustness table column count + multicolumn header
#   e0ae72d (article.tex): \usepackage{threeparttable}

def patch_file(path, transforms, name):
    text = path.read_text(encoding='utf-8', errors='ignore')
    original = text
    applied = 0
    for needle, replacement in transforms:
        if replacement in text:
            continue  # already patched
        if needle not in text:
            print(f'  [skip] {name}: not found: {needle[:50]!r}')
            continue
        text = text.replace(needle, replacement, 1)
        applied += 1
    if text != original:
        path.write_text(text, encoding='utf-8')
        print(f'Patched {name}: {applied} transformation(s) applied')
    else:
        print(f'{name}: no changes needed (already up-to-date)')

memoire_path = LATEX_DIR / 'memoire.tex'
if memoire_path.exists():
    sect_title = "\\section{Module de Fusion par Attention Crois\\'{e}e Modul\\'{e}e par la Qualit\\'{e}}"
    memoire_patches = [
        ("\\usepackage{algpseudocode}",
         "\\usepackage{algorithmic}"),
        ("\\usepackage{etoolbox}\n\n% Fix abstract package conflict with French babel",
         "\\usepackage{etoolbox}\n\n% Custom commands\n\\newcommand{\\TO}{\\textbf{to}}\n\\newcommand{\\margin}{\\ensuremath{m}}\n\n% Fix abstract package conflict with French babel"),
        (sect_title + "\n\\label{sec:attention}",
         sect_title + "\n\\label{sec:fusion-attention}"),
        ("(\\S\\ref{sec:attention})",
         "(\\S\\ref{sec:fusion-attention})"),
        ("\\label{tab:resultats-principaux}\n    \\begin{tabular}{lcccccc}",
         "\\label{tab:resultats-principaux}\n    \\begin{tabular}{lccccc}"),
    ]
    patch_file(memoire_path, memoire_patches, 'memoire.tex')

article_path = LATEX_DIR / 'article.tex'
if article_path.exists():
    article_patches = [
        ("\\usepackage{pifont}\n\n\\begin{document}",
         "\\usepackage{pifont}\n\\usepackage{threeparttable}\n\n\\begin{document}"),
        ("\\begin{tabular}{l | c | c c c c c | c c c}",
         "\\begin{tabular}{l | c | c c c c c c | c c c}"),
        ("& & & & & & & \\multicolumn{3}{c}{\\textbf{Single Modality}} \\\\",
         "& & & & & & & & \\multicolumn{3}{c}{\\textbf{Single Modality}} \\\\"),
    ]
    patch_file(article_path, article_patches, 'article.tex')

print()
print('Files present in LATEX_DIR:')
for p in sorted(LATEX_DIR.glob('*.tex')):
    print(f'  - {p.name} ({p.stat().st_size} bytes)')


In [ ]:
has_pdflatex = shutil.which('pdflatex') is not None
has_bibtex = shutil.which('bibtex') is not None
print('pdflatex available:', has_pdflatex)
print('bibtex available:', has_bibtex)

import subprocess as _sp, time as _time
PASS_TIMEOUT = 300  # seconds per pdflatex/bibtex pass; >0 prevents infinite hangs

def _run_tex(cmd, cwd, log_name):
    """Run one tex pass with timeout + streaming. Capture to log file."""
    log_path = LATEX_DIR / log_name
    print(f'  [{" ".join(cmd)}]  ', end='', flush=True)
    t0 = _time.time()
    try:
        with open(log_path, 'w') as logf:
            proc = _sp.Popen(cmd, cwd=cwd, stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True, encoding='latin-1', errors='replace')
            try:
                for line in proc.stdout:
                    print(line, end='')
                    logf.write(line)
                rc = proc.wait(timeout=PASS_TIMEOUT)
            except _sp.TimeoutExpired:
                proc.kill()
                proc.wait()
                print(f'TIMEOUT after {PASS_TIMEOUT}s', flush=True)
                logf.write(f'\n*** TIMEOUT after {PASS_TIMEOUT}s ***\n')
                return -1
        elapsed = _time.time() - t0
        print(f'done in {elapsed:.1f}s (rc={rc})')
        return rc
    except FileNotFoundError:
        print('binary not found — skipping')
        return -1

def compile_tex(doc_name: str):
    if not has_pdflatex:
        print(f'Skipping {doc_name}: pdflatex not available')
        return
    print(f'\n=== Compiling {doc_name}.tex ===')
    _run_tex(['pdflatex', '-interaction=nonstopmode', f'{doc_name}.tex'], str(LATEX_DIR), f'{doc_name}.pass1.log')
    if has_bibtex:
        _run_tex(['bibtex', doc_name], str(LATEX_DIR), f'{doc_name}.bibtex.log')
    _run_tex(['pdflatex', '-interaction=nonstopmode', f'{doc_name}.tex'], str(LATEX_DIR), f'{doc_name}.pass2.log')
    _run_tex(['pdflatex', '-interaction=nonstopmode', f'{doc_name}.tex'], str(LATEX_DIR), f'{doc_name}.pass3.log')
    pdf = LATEX_DIR / f'{doc_name}.pdf'
    if pdf.exists():
        print(f'  OK: {pdf.name} ({pdf.stat().st_size} bytes)')
    else:
        print(f'  !! {pdf.name} NOT produced — see {LATEX_DIR}/{doc_name}.*.log')

compile_tex('article')
compile_tex('memoire')


## 10) Final artifact summary

In [ ]:
def show_files(title, root: Path, patterns):
    print('\n' + title)
    if not root.exists():
        print('  (missing)')
        return
    found = []
    for pat in patterns:
        found.extend(root.glob(pat))
    found = sorted(set(found))
    if not found:
        print('  (none found)')
    else:
        for p in found:
            print(' -', p)

show_files('Model checkpoints', OUTPUT_DIR, ['**/*.pth'])
show_files('Evaluation/visualization outputs', RESULTS_DIR, ['**/*.json', '**/*.pdf', '**/*.tex'])
show_files('LaTeX artifacts', LATEX_DIR, ['*.tex', '*.pdf', 'figures/*.pdf', 'tables/*.tex'])


In [ ]:
import zipfile
from pathlib import Path
import os

# Zip each top-level folder under /kaggle/working into its own .zip
working_dir = Path('/kaggle/working')

# Skip 'output' (model checkpoints are large and not needed downstream)
EXCLUDE = {'output', '.ipynb_checkpoints'}

for entry in sorted(working_dir.iterdir()):
    if not entry.is_dir() or entry.name.startswith('.') or entry.name in EXCLUDE:
        continue

    archive_path = working_dir / f'{entry.name}.zip'
    print(f'\nZipping {entry.name}/ -> {archive_path.name} ...')

    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(entry):
            dirs[:] = [d for d in dirs if not d.startswith('.')]
            for file in files:
                file_path = Path(root) / file
                arcname = file_path.relative_to(entry.parent)
                zf.write(file_path, arcname)

    size_mb = archive_path.stat().st_size / 1024 / 1024
    print(f'  Created: {archive_path.name} ({size_mb:.1f} MB)')

print('\nAll zips:')
for z in sorted(working_dir.glob('*.zip')):
    mb = z.stat().st_size / 1024 / 1024
    print(f'  {z.name}  ({mb:.1f} MB)')
